# Data Profiling

## 1. Datensatz (Überblick)

Der Datensatz liegt als Excel-Arbeitsmappe mit zwei Arbeitsblättern vor. Bevor das eigentliche Profiling beginnt, wird geprüft, ob beide Arbeitsblätter gemeinsam analysiert werden können.

In [ ]:
from paths import RAW_DATA_DIR
import pandas as pd

xlsx_file = RAW_DATA_DIR / "online_retail_II.xlsx"

sheet_1 = pd.read_excel(xlsx_file, sheet_name="Year 2009-2010")
sheet_2 = pd.read_excel(xlsx_file, sheet_name="Year 2010-2011")

## 2. Struktur der Arbeitsblätter

Die Struktur beider Arbeitsblätter wird verglichen. Dazu werden die Spaltennamen, deren Reihenfolge sowie die Datentypen betrachtet.

In [ ]:
print([sheet.info() for sheet in [sheet_1, sheet_2]])

Beide Arbeitsblätter besitzen dieselben Spalten in identischer Reihenfolge sowie kompatible Datentypen.
Daher werden sie für das weitere Profiling zu einem gemeinsamen DataFrame zusammengeführt.

In [ ]:
df = pd.concat([sheet_1, sheet_2], ignore_index=True)

## 3. Spaltenanalyse

Die Spalten werden hinsichtlich ihrer Datentypen, fehlender Werte und möglicher Identifikatoren untersucht.
Offensichtliche Importartefakte werden vor den weiteren Analysen korrigiert.

In [ ]:
print(df.info())

Die Spalten **Invoice** und **StockCode** enthalten Identifikatoren, **Description** Textinformationen.
Diese Spalten werden für die weitere Analyse in den String-Datentyp konvertiert.
Dadurch bleibt der Informationsgehalt unverändert, während Vergleiche und Sortierungen konsistent durchgeführt werden können.

In [ ]:
df["Invoice"] = df["Invoice"].astype("string")
df["StockCode"] = df["StockCode"].astype("string")
df["Description"] = df["Description"].astype("string")

print(df.info())

Die Datentypen entsprechen nun dem Inhalt der Spalten und bilden die Grundlage für die folgenden Analysen.

In [ ]:
print(df.isna().sum(), end="\n\n")
print(f"Invoice (is unique): {df["Invoice"].is_unique}\n")
print(f"[Invoice, StockCode] (is unique): {len(df[["Invoice", "StockCode"]].drop_duplicates()) == len(df)}")

Die Spalten **Description** und **Customer ID** enthalten fehlende Werte.
Weder **Invoice** noch die Kombination aus **Invoice und StockCode** identifizieren eine Zeile eindeutig.
Dasselbe Produkt kann innerhalb einer Rechnung mehrfach als separate Position vorkommen, obwohl die Menge in **Quantity** erfasst wird.
Ein eindeutiger Schlüssel auf Positionsebene ist nicht vorhanden.

In [ ]:
print(df["Invoice"].value_counts())
print(df["Invoice"].value_counts().mean())
print(df["Invoice"].value_counts().median())

counts = df["Invoice"].value_counts()
for idx in counts[counts > 1000].index:
    idx_loc = df["Invoice"] == idx
    print(df.loc[idx_loc, "Customer ID"].count())
    print(df.loc[idx_loc, "Country"].unique())
    print((df.loc[idx_loc, "Quantity"] < 0).sum())
    print((df.loc[idx_loc, "Price"] < 0).sum())

display(df[df["Invoice"] == 537434].sort_values(by="StockCode").head(30))

display(
    df[df.duplicated(keep=False)]
    # df[df.duplicated(keep="first")]
    .sort_values(by=["Invoice", "StockCode"])
)

Die Anzahl der Positionen pro Rechnung variiert deutlich.
Während die meisten Rechnungen nur wenige Positionen enthalten, existieren einzelne Rechnungen mit mehr als 1.000 Positionen.

Negative Mengen oder Preise könnten auf Storno-, Rückgabe- oder Korrekturbuchungen hindeuten.
Die untersuchten Rechnungen mit mehr als 1.000 Positionen weisen jedoch weder negative Mengen noch negative Preise auf.
Darüber hinaus enthalten sie keine Customer ID, stammen ausschließlich aus dem Vereinigten Königreich und beginnen nicht mit dem Präfix „C“, das im Datensatz
für Stornorechnungen verwendet wird. Die fachliche Bedeutung dieser Rechnungen lässt sich anhand der verfügbaren Informationen nicht eindeutig bestimmen.

Bei der Detailanalyse dieser Rechnungen fallen vollständig identische Rechnungspositionen auf. Eine anschließende Untersuchung des gesamten Datensatzes zeigt,
dass 67.242 Zeilen Teil vollständig identischer Datensätze sind. Davon stellen 34.335 Zeilen redundante Wiederholungen dar.

Da diese Zeilen in sämtlichen Spalten identisch sind, enthalten die zusätzlichen Wiederholungen keine weiteren Informationen.
Sie können daher im Rahmen der späteren Datenbereinigung entfernt werden.

## Zusammenhänge zwischen Invoice und Quantity

Die mit `C` beginnenden Rechnungen werden hinsichtlich ihrer Mengen untersucht.

In [ ]:
from paths import RAW_DATA_DIR

cancellations = df[df["Invoice"].str.startswith("C")]
print((cancellations["Quantity"] < 0).value_counts())
print(cancellations[(cancellations["Quantity"] > 0)])
print(len(cancellations))

# cancellations.to_csv(
#     RAW_DATA_DIR / "cancellations.csv",
#     index=False
# )

Laut Datensatzbeschreibung kennzeichnet das Präfix `C` Stornierungen. Die Untersuchung bestätigt diese Annahme weitgehend.
Nahezu alle betroffenen Rechnungspositionen besitzen eine negative Menge.

Es existiert jedoch mindestens eine Ausnahme mit einer positiven Menge.
Die zugehörige Buchung besitzt den **StockCode** `M`, die Beschreibung `Manual` sowie keine **Customer ID**.
Die fachliche Bedeutung dieses Datensatzes kann anhand der verfügbaren Informationen nicht eindeutig bestimmt werden.

Darüber hinaus treten negative Mengen auch bei Rechnungen ohne `C`-Präfix auf.
Eine negative Menge kann daher nicht ausschließlich über das Rechnungspräfix erklärt werden.

In [ ]:
print(f"cancellation_pct: {len(cancellations) * 100 / len(df)}%")

Insgesamt entfallen rund **1,83 %** aller Datensätze auf Rechnungen mit dem Präfix `C`.

## Untersuchung alphanumerischer StockCodes

Zunächst werden **StockCodes** betrachtet, die mit einer Ziffer beginnen und mit einem Buchstaben enden.

In [ ]:
from paths import RAW_DATA_DIR
import regex as re

postfix = df[df["StockCode"].str.match(r"^\d+[a-z]", flags=re.IGNORECASE)]
print(postfix.head(10))
print(len(postfix))

# postfix.to_csv(
#     RAW_DATA_DIR / "postfix.csv",
#     index=False
# )

Diese Datensätze wurden zusätzlich exportiert und manuell untersucht.

Die manuelle Durchsicht zeigt, dass der angehängte Buchstabe überwiegend Bestandteil regulärer Produktcodes ist und nicht allgemein zur Kennzeichnung von Sonderfällen dient.

Innerhalb dieser Gruppe treten zwar auch negative Mengen, fehlende Beschreibungen und weitere Auffälligkeiten auf, diese betreffen jedoch nur einen kleinen Teil der Datensätze.
Von insgesamt **128.893** Zeilen besitzen lediglich **3.377** eine negative Menge.

In [ ]:
print(f"postfix_pct: {len(postfix) * 100 / len(df)}%")

Die Gruppe umfasst rund **12,08 %** des gesamten Datensatzes. Aufgrund der manuellen Untersuchung werden diese **StockCodes** nicht grundsätzlich als Sonderfälle behandelt.

## Untersuchung nicht numerisch beginnender StockCodes

Anschließend werden **StockCodes** untersucht, die nicht mit einer Ziffer beginnen.

In [ ]:
from paths import RAW_DATA_DIR
import regex as re

business_cases = df[df["StockCode"].str.match(r"^[^\d]+", flags=re.IGNORECASE)]

print(len(business_cases))
print(business_cases["StockCode"].value_counts().head(10))

# business_cases.to_csv(
#     RAW_DATA_DIR / "business_cases.csv",
#     index=False
# )

Auch diese Datensätze wurden exportiert und manuell untersucht.

Insgesamt enthält der Datensatz **64** unterschiedliche **StockCodes**, die nicht mit einer Ziffer beginnen.
Die häufigsten Codes sind unter anderem `POST`, `DOT`, `M`, `D`, `BANK CHARGES`, `ADJUST` und `AMAZONFEE`.

Die zugehörigen Beschreibungen deuten darauf hin, dass ein Teil dieser Datensätze administrative Geschäftsvorgänge wie Versandkosten, Rabatte, Gebühren oder manuelle Buchungen repräsentiert.
Der verwendete Filter liefert jedoch keine eindeutige Trennung zwischen Produkttransaktionen und administrativen Buchungen.
Einige der gefundenen Codes scheinen reguläre Produkte oder Gutscheine zu beschreiben.

In [135]:
print(f"business_cases_pct: {len(business_cases) * 100 / len(df)}%")

business_cases_pct: 0.5708418160133637%


## Fazit der Spaltenanalyse

Neben regulären Produkttransaktionen enthält der Datensatz verschiedene Sonderfälle wie Stornierungen, Gebühren, Rabatte, Versandkosten und manuelle Buchungen.

Viele dieser Erkenntnisse konnten nicht allein durch aggregierte Auswertungen gewonnen werden.
Sie beruhen auf der gezielten manuellen Untersuchung exportierter Teilmengen des Datensatzes.

Die Untersuchung zeigt außerdem, dass die beobachteten Sonderfälle nur einen vergleichsweise kleinen Anteil des Gesamtdatensatzes ausmachen.
Während Rechnungen mit dem Präfix `C` rund **1,83 %** aller Datensätze umfassen und nicht numerisch beginnende **StockCodes** etwa **0,57 %**,
stellen **StockCodes** mit angehängtem Buchstaben zwar rund **12,08 %** des Datensatzes dar, repräsentieren jedoch überwiegend reguläre Produktcodes.

Eine vollständige fachliche Einordnung sämtlicher Buchungen ist anhand des Datensatzes allein dennoch nicht möglich.
Hierfür wären zusätzliche Dokumentationen des Quellsystems oder die Unterstützung eines Domainexperten erforderlich.

Für eine klar definierte Businessfrage lässt sich jedoch eine nachvollziehbare Datenaufbereitung entwickeln.
Die späteren Bereinigungsregeln sollten deshalb ausschließlich diejenigen Datensätze ausschließen,
deren Ausschluss für den jeweiligen Analysezweck fachlich begründet werden kann.